In [3]:
import subprocess
import shlex
import sys
import os
import oxbow
import polars as pl
from rich.console import Console

os.chdir("/zata/zippy/ramirezc/atac-smk")

In [4]:
input_bam = "test/EA100050-GRCH38.bam"
# threads = 16

# console = Console()

# with console.status("[cyan]Running PBC QC metric getter...", spinner="dots8", speed=2) as status:
#     samtools_result = subprocess.run(shlex.split(f"samtools sort --threads {threads} -n {input_bam}"), stdout=subprocess.PIPE)
#     console.log("Finished samtools")
    
#     bedtools_result = subprocess.run(["bedtools", "bamtobed", "-i", "stdin"], input=samtools_result.stdout, stdout=subprocess.PIPE)
#     console.log("Finished bedtools")
    
#     awk_results = subprocess.run(shlex.split(r"""awk 'BEGIN{OFS="\t"}{print $1,$2,$4,$6,$9,$10}'"""), input=bedtools_result.stdout, stdout=subprocess.PIPE)
#     console.log("Finished awk")
    
#     grep_results = subprocess.run(shlex.split("grep -E -v 'chrM|chrUn_'"), input=awk_results.stdout, stdout=subprocess.PIPE)
#     console.log("Finished grep")
    
#     sort_results = subprocess.run(["sort"], input=grep_results.stdout, stdout=subprocess.PIPE)
#     console.log("Finished sort")
    
#     unique_results = subprocess.run(shlex.split("uniq -c"), input=sort_results.stdout, stdout=subprocess.PIPE)
#     console.log("Finished unique")
    
#     awk_results2 = subprocess.run(shlex.split(r"""awk 'BEGIN{mt=0;m0=0;m1=0;m2=0} ($1==1){m1=m1+1} ($1==2){m2=m2+1} {m0=m0+1} {mt=mt+$1} END{printf "%d\t%d\t%d\t%d\t%f\t%f\t%f\n",mt,m0,m1,m2,m0/mt,m1/m0,m1/m2}'"""), input=unique_results.stdout, stdout=subprocess.PIPE)
#     console.log("Finished awk2")
    
#     console.log(awk_results2.stdout.decode('utf-8'))


# # with open(.pbc_txt, "wb") as f:
# #     f.write(awk_results2.stdout)

# # logfile.close()

In [5]:
bam_file = oxbow.from_bam(input_bam).to_polars(lazy=True)
display(bam_file.head(10).collect().to_dicts()[2])
print(bam_file.select(pl.len()).collect().item())

{'qname': 'AV231102:run99:2435416092:1:10503:0266:0043',
 'flag': 419,
 'rname': 'chr1',
 'pos': 10004,
 'mapq': None,
 'cigar': '76M',
 'rnext': 'chr1',
 'pnext': 10027,
 'tlen': 99,
 'seq': 'CCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTCACCCTAACCCTAATCCCAACCCC',
 'qual': ':EGI6IAIMF=3M+K@*M5N=A4K6FM7-N=&M>(H56N*<7MFN/H8KHHD-5J99$)LGAN5&=%KF)5;8FM+',
 'end': 10079,
 'tags': {'AS': -10,
  'MD': '52A13C2T5T0',
  'NM': 4,
  'RG': 'EA100050-GRCH38',
  'XG': 0,
  'XM': 4,
  'XN': 0,
  'XO': 0,
  'XS': -10,
  'YS': 0,
  'YT': 'CP'}}

66403118


In [19]:
# flag = 163
# b = [flag >> i & 1 for i in range(flag.bit_length() - 1,-1,-1)]
# print(b)

Only when we want a non-bedpe file:

```python
pl.when(pl.col("flag") & 64 == 0)
.then(pl.col("qname") + "/1")
.when(pl.col("flag") & 128 == 0)
.then(pl.col("qname") + "/2")
.otherwise(pl.col("qname"))
.alias("name")
```

In [20]:
bed_cols = ["chrom", "start", "end", "name", "score", "strand"]
agg_cols = ["chrom", "start", "end", "strand"]

console = Console()

with console.status("[cyan]Running PBC QC metric getter...", spinner="dots8", speed=2, refresh_per_second=25) as status:
    bed_lf = bam_file.with_columns(
        pl.col("rname").alias("chrom"),
        pl.min_horizontal("pos", "pnext").alias("start"),
        (pl.max_horizontal("pos", "pnext") + pl.col("tlen")).alias("end"),
        pl.col("mapq").alias("score"),
        pl.col("qname").alias("name"),
        pl.when(pl.col("flag") & 16 == 0)
        .then(pl.lit("+"))
        .otherwise(pl.lit("-"))
        .alias("strand")
    ).select(bed_cols + ["flag"]).filter(
        ~((pl.col("chrom") == "chrM") | (pl.col("chrom").cat.starts_with("chrUn"))), # Filter mitochondrial and reads mapped to contigs/scaffolds
        ((pl.col("flag") & 1804 == 0) & (pl.col("flag") & 2 != 0)), # filtering for (same as filter.smk)
        (pl.col("score") >= 30)
    ).group_by("name").agg(
        [pl.col(col).filter(pl.col("flag") & 64 == 0).first().alias(col) for col in agg_cols] + \
        [pl.col(col).filter(pl.col("flag") & 128 == 0).first().alias(col + "_") for col in agg_cols] + \
        [pl.col("score").first()]
    ).select(
        ["chrom", "start", "end", "chrom_", "start_", "end_", "name", "score", "strand", "strand_"]
    ).sort("name")
    
    bed_df = bed_lf.collect()
    display(bed_df)

uniq_counts = bed_df.with_columns(
    pl.concat_str(["chrom", "start", "end", "chrom_", "start_", "end_"]).alias("uniq")
)["uniq"].value_counts()

display(bed_df)

Output()

chrom,start,end,chrom_,start_,end_,name,score,strand,strand_
cat,i32,i32,cat,i32,i32,str,u8,str,str
"""chr17""",27066210,27066174,"""chr17""",27066210,27066246,"""AV231102:run99:2435416092:1:10…",35,"""-""","""+"""
"""chr11""",97727625,97727680,"""chr11""",97727625,97727570,"""AV231102:run99:2435416092:1:10…",30,"""+""","""-"""
"""chr2""",198215247,198215287,"""chr2""",198215247,198215207,"""AV231102:run99:2435416092:1:10…",31,"""+""","""-"""
"""chr3""",156224078,156224004,"""chr3""",156224078,156224152,"""AV231102:run99:2435416092:1:10…",37,"""-""","""+"""
"""chr17""",42032142,42032066,"""chr17""",42032142,42032428,"""AV231102:run99:2435416092:1:10…",37,"""-""","""+"""
…,…,…,…,…,…,…,…,…,…
"""chr9""",93588624,93588548,"""chr9""",93588624,93589026,"""AV231102:run99:2435416092:2:22…",37,"""-""","""+"""
"""chr4""",78116022,78115946,"""chr4""",78116022,78116298,"""AV231102:run99:2435416092:2:22…",32,"""-""","""+"""
"""chr20""",56500120,56500444,"""chr20""",56500120,56500044,"""AV231102:run99:2435416092:2:22…",35,"""+""","""-"""


chrom,start,end,chrom_,start_,end_,name,score,strand,strand_
cat,i32,i32,cat,i32,i32,str,u8,str,str
"""chr17""",27066210,27066174,"""chr17""",27066210,27066246,"""AV231102:run99:2435416092:1:10…",35,"""-""","""+"""
"""chr11""",97727625,97727680,"""chr11""",97727625,97727570,"""AV231102:run99:2435416092:1:10…",30,"""+""","""-"""
"""chr2""",198215247,198215287,"""chr2""",198215247,198215207,"""AV231102:run99:2435416092:1:10…",31,"""+""","""-"""
"""chr3""",156224078,156224004,"""chr3""",156224078,156224152,"""AV231102:run99:2435416092:1:10…",37,"""-""","""+"""
"""chr17""",42032142,42032066,"""chr17""",42032142,42032428,"""AV231102:run99:2435416092:1:10…",37,"""-""","""+"""
…,…,…,…,…,…,…,…,…,…
"""chr9""",93588624,93588548,"""chr9""",93588624,93589026,"""AV231102:run99:2435416092:2:22…",37,"""-""","""+"""
"""chr4""",78116022,78115946,"""chr4""",78116022,78116298,"""AV231102:run99:2435416092:2:22…",32,"""-""","""+"""
"""chr20""",56500120,56500444,"""chr20""",56500120,56500044,"""AV231102:run99:2435416092:2:22…",35,"""+""","""-"""


In [21]:
no_total_reads = uniq_counts["count"].sum()
no_distict_reads = uniq_counts.height
no_unique_reads = uniq_counts.filter(pl.col("count") == 1).height
no_reads_seen_twice = uniq_counts.filter(pl.col("count") == 2).height

print(f"Total reads: {no_total_reads}")
print(f"Distinct reads: {no_distict_reads}")
print(f"Unique reads: {no_unique_reads}")
print(f"Reads seen twice: {no_reads_seen_twice}")

Total reads: 1556720
Distinct reads: 1520668
Unique reads: 1485863
Reads seen twice: 33925


In [22]:
PBC1 = no_total_reads / no_distict_reads
if no_reads_seen_twice != 0:
    PBC2 = no_total_reads / no_reads_seen_twice
else:
    PBC2 = float('inf')
NRF = no_distict_reads / no_total_reads

print(f"PBC1: {PBC1}")
print(f"PBC2: {PBC2}")
print(f"NRF: {NRF}")

PBC1: 1.0237080020096432
PBC2: 45.88710390567428
NRF: 0.9768410504136903


In [8]:
# MAPQ distribution before filtering
bam_file.group_by("mapq").len().sort("mapq").collect().drop_nulls().plot.bar(x="mapq:O", y="len:Q")

alt.Chart(...)